<a href="https://colab.research.google.com/github/shireinnn/proyekbigdata/blob/main/ProyekkBigData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Setup PySpark di Google Colab

Langkah pertama adalah menginstal library `pyspark` dan inisialisasi sesi Spark.

In [1]:
# Uninstall potentially conflicting packages and then install PySpark and findspark (specific version for compatibility)
!pip uninstall -y dataproc-spark-connect
!pip install pyspark==3.5.0 findspark

import os
import findspark
import sys

# Clean up previous Spark installation if any
!rm -rf /content/spark
!rm -rf spark-3.5.0-bin-hadoop3.tgz

# Download and extract Spark
# Using Spark 3.5.0 with Hadoop 3.3 for compatibility with PySpark 3.5.0
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!mv spark-3.5.0-bin-hadoop3 /content/spark

# Ensure Java 11 is used
!sudo apt-get update -qq > /dev/null
!sudo apt-get install openjdk-11-jdk-headless -qq > /dev/null
!sudo update-alternatives --set java /usr/lib/jvm/java-11-openjdk-amd64/bin/java

# Set JAVA_HOME and SPARK_HOME environment variables
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark"

# Explicitly set PYSPARK_PYTHON and PYSPARK_SUBMIT_ARGS
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_SUBMIT_ARGS"] = "--master local[*] pyspark-shell"

# Initialize findspark to configure environment variables
findspark.init()

print("PySpark dan findspark berhasil diinstal dan variabel lingkungan diatur.")
print("SPARK_HOME:", os.environ.get("SPARK_HOME"))
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("PYSPARK_PYTHON:", os.environ.get("PYSPARK_PYTHON"))
print("PYSPARK_SUBMIT_ARGS:", os.environ.get("PYSPARK_SUBMIT_ARGS"))

Found existing installation: dataproc-spark-connect 1.1.0
Uninstalling dataproc-spark-connect-1.1.0:
  Successfully uninstalled dataproc-spark-connect-1.1.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 1.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 11.9 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.5.0-py2.py3-none-any.whl size=317425346 sha256=091f761093d84feab5adac82fdd9fc6b130b0eaae54d987bf28020273d3ab660
  Stored in directory: /root/.cache/pip/wheels/84/40/20/65eefe766118e0a8f8e385cc3ed6e9eb7241c7e51cfc04c51a
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.0.2
    Uninstalling pyspark-4.0.2:
      Successfully uninstalled pyspark-4.0.2
W: Skipping acquire of configured fil

In [2]:
from pyspark.sql import SparkSession

# Create a SparkSession
spark = SparkSession.builder\
    .appName("MarketBasketAnalysis")\
    .getOrCreate()

print("SparkSession berhasil dibuat!")

SparkSession berhasil dibuat!


### 2. Load File

Mengekstrak file ZIP yang berisi data Parquet dan memuatnya ke dalam DataFrame Spark.PySpark dapat membaca semua file Parquet dalam direktori ini secara rekursif.

In [4]:
# Import library yang dibutuhkan untuk ekstraksi
import zipfile
import os

# Path ke file ZIP Anda
zip_file_path = "/content/sample_data/basket_transactions.zip"

# Direktori tujuan untuk mengekstrak file Parquet
extraction_path = "/content/basket_transactions"

# Pastikan direktori tujuan ada
os.makedirs(extraction_path, exist_ok=True)

# Ekstrak file ZIP
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)

print(f"File '{zip_file_path}' berhasil diekstrak ke '{extraction_path}'")

# Path ke folder Parquet yang sudah diekstrak
parquet_path = extraction_path

# Load data Parquet
transactions_df = spark.read.parquet(parquet_path)

# Tampilkan skema dan beberapa baris pertama data
transactions_df.printSchema()
transactions_df.show(5, truncate=False)

# Periksa jumlah baris
print(f"Total baris dalam dataset: {transactions_df.count()}")

File '/content/sample_data/basket_transactions.zip' berhasil diekstrak ke '/content/basket_transactions'
root
 |-- Transaction_ID: string (nullable = true)
 |-- Product: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- Discount_Applied: integer (nullable = true)
 |-- Season: string (nullable = true)

+--------------+-------------------------------------------------------------+----------------+------+
|Transaction_ID|Product                                                      |Discount_Applied|Season|
+--------------+-------------------------------------------------------------+----------------+------+
|1000000014    |[Razors, Laundry Detergent, Beef]                            |1               |Spring|
|1000000016    |[Air Freshener, Feminine Hygiene Products]                   |1               |Spring|
|1000000027    |[Ketchup, Trash Cans, Toothbrush]                            |0               |Spring|
|1000000035    |[Onions, Dish Soap]                 

### 3. Jalankan FP-Growth


In [7]:
from pyspark.ml.fpm import FPGrowth
from pyspark.sql.functions import array_distinct

# Preprocess the 'Product' column to ensure unique items in each transaction array
# This is necessary because FPGrowth requires unique items within each transaction
transactions_df_cleaned = transactions_df.withColumn("Product", array_distinct(transactions_df["Product"]))

# --- Menganalisis transaksi dengan diskon (Discount_Applied == 1) ---
print("\n--- Running FP-Growth for transactions with Discount_Applied == 1 ---")

transactions_with_discount = transactions_df_cleaned.filter(transactions_df_cleaned["Discount_Applied"] == 1)

# Inisialisasi model FP-Growth untuk transaksi berdiskon dengan minSupport yang lebih rendah dan minConfidence yang sangat rendah
fpg_discounted = FPGrowth(itemsCol="Product", minSupport=0.001, minConfidence=0.01)

# Fit model ke data transaksi yang sudah dibersihkan dan berdiskon
model_discounted = fpg_discounted.fit(transactions_with_discount)

# Dapatkan frequent itemsets untuk transaksi berdiskon
frequent_itemsets_discounted = model_discounted.freqItemsets
print("Frequent Itemsets (Discounted Transactions - minSupport=0.001, minConfidence=0.01):")
frequent_itemsets_discounted.show(20, truncate=False)

# Dapatkan association rules untuk transaksi berdiskon
association_rules_discounted = model_discounted.associationRules
print("Association Rules (Discounted Transactions - minSupport=0.001, minConfidence=0.01):")
association_rules_discounted.show(20, truncate=False)


--- Running FP-Growth for transactions with Discount_Applied == 1 ---
Frequent Itemsets (Discounted Transactions - minSupport=0.001, minConfidence=0.01):
+-----------------------+-----+
|items                  |freq |
+-----------------------+-----+
|[Apple]                |16762|
|[Apple, Tomatoes]      |576  |
|[Apple, BBQ Sauce]     |588  |
|[Apple, Honey]         |573  |
|[Apple, Garden Hose]   |607  |
|[Apple, Chips]         |584  |
|[Apple, Tissues]       |544  |
|[Apple, Cleaning Rags] |576  |
|[Apple, Vacuum Cleaner]|595  |
|[Apple, Dish Soap]     |615  |
|[Apple, Mop]           |573  |
|[Apple, Olive Oil]     |585  |
|[Apple, Bath Towels]   |580  |
|[Apple, Cleaning Spray]|570  |
|[Apple, Ice Cream]     |593  |
|[Apple, Soap]          |597  |
|[Apple, Banana]        |592  |
|[Apple, Salmon]        |562  |
|[Apple, Rice]          |610  |
|[Apple, Tea]           |582  |
+-----------------------+-----+
only showing top 20 rows

Association Rules (Discounted Transactions - minSup

### 4. Perbandingan Metrik Aturan Asosiasi (Diskon vs. Tanpa Diskon)

In [33]:
import pandas as pd

comparison_data = []

for (season, discount_applied), rules_df in high_lift_rules_by_segment.items():
    if rules_df is not None and rules_df.count() > 0:
        num_rules = rules_df.count()
        avg_confidence = rules_df.selectExpr("avg(confidence)").collect()[0][0]
        avg_lift = rules_df.selectExpr("avg(lift)").collect()[0][0]
        avg_support = rules_df.selectExpr("avg(support)").collect()[0][0]
    else:
        num_rules = 0
        avg_confidence = None
        avg_lift = None
        avg_support = None

    comparison_data.append({
        'Season': season,
        'Discount_Applied': 'Yes' if discount_applied == 1 else 'No',
        'Number_of_Rules': num_rules,
        'Avg_Confidence': avg_confidence,
        'Avg_Lift': avg_lift,
        'Avg_Support': avg_support
    })

# Create a Pandas DataFrame for better visualization
comparison_df = pd.DataFrame(comparison_data)

# Sort for better readability
comparison_df = comparison_df.sort_values(by=['Season', 'Discount_Applied'], ascending=[True, False])

print("\nPerbandingan Metrik Aturan Asosiasi (Lift > 1.0) Berdasarkan Musim dan Diskon:")
print(comparison_df.to_string())


Perbandingan Metrik Aturan Asosiasi (Lift > 1.0) Berdasarkan Musim dan Diskon:
   Season Discount_Applied  Number_of_Rules  Avg_Confidence  Avg_Lift  Avg_Support
4    Fall              Yes               24        0.042357  1.011456     0.001774
5    Fall               No               20        0.042346  1.012528     0.001771
0  Spring              Yes               16        0.042909  1.024960     0.001796
1  Spring               No               20        0.042709  1.021108     0.001786
2  Summer              Yes               22        0.043035  1.022380     0.001811
3  Summer               No               30        0.042673  1.018526     0.001788
6  Winter              Yes               22        0.042971  1.024906     0.001801
7  Winter               No               32        0.042791  1.019381     0.001796


### 10. Mengidentifikasi Frequent Itemsets yang Umum di Seluruh Musim dan Status Diskon

Untuk memahami pola pembelian yang mendasar di seluruh siklus bisnis, kita akan mengidentifikasi *frequent itemsets* yang muncul di *semua* kombinasi musim (`Spring`, `Summer`, `Fall`, `Winter`) dan status diskon (`Discount_Applied = 1` atau `0`). Ini akan menunjukkan produk atau kombinasi produk yang secara universal populer atau dibeli bersama.

Langkah-langkahnya adalah:
1. Mengumpulkan semua *frequent itemsets* untuk setiap segmen (musim + status diskon).
2. Menemukan irisan (*intersection*) dari semua *frequent itemsets* yang dikumpulkan.

### Frequent Itemsets Multi-Item untuk Semua Segmen dengan Aturan Asosiasi Lift > 1.0

In [37]:
from pyspark.ml.fpm import FPGrowth
from pyspark.sql.functions import col, size, array_contains

print("Menampilkan frequent itemsets multi-item (tanpa 'Toothpaste') untuk setiap segmen dengan aturan lift > 1.0:")

# Iterate through the segments that had high lift rules
for (season, discount_applied), rules_df in high_lift_rules_by_segment.items():
    if rules_df is not None and rules_df.count() > 0:
        discount_status_str = 'Yes' if discount_applied == 1 else 'No'
        print(f"\n--- Segmen: Musim {season}, Diskon: {discount_status_str} ---")

        # Filter transactions for the current segment
        segment_transactions = transactions_df_cleaned.filter(
            (transactions_df_cleaned['Season'] == season) &
            (transactions_df_cleaned["Discount_Applied"] == discount_applied)
        )

        if segment_transactions.count() > 0:
            # Re-fit FP-Growth to get frequent itemsets with frequencies
            # Use the same minSupport and minConfidence that led to high lift rules
            fpg_segment = FPGrowth(itemsCol="Product", minSupport=0.001, minConfidence=0.01)
            model_segment = fpg_segment.fit(segment_transactions)
            frequent_itemsets_segment = model_segment.freqItemsets

            # Filter for itemsets with 2 or more items AND exclude those containing 'Toothpaste'
            multi_item_frequent_itemsets = frequent_itemsets_segment.filter(
                (size(col("items")) >= 2) & (~array_contains(col("items"), "Toothpaste"))
            )

            if multi_item_frequent_itemsets.count() > 0:
                print(f"Frequent Itemsets dengan 2 atau Lebih Produk (Tanpa 'Toothpaste', Top 10):")
                multi_item_frequent_itemsets.orderBy(col("freq").desc()).show(10, truncate=False)
            else:
                print("  Tidak ada frequent itemsets dengan 2 atau lebih produk (tanpa 'Toothpaste') ditemukan untuk segmen ini.")
        else:
            print("  Tidak ada transaksi untuk segmen ini. Melewatkan.")
    else:
        discount_status_str = 'Yes' if discount_applied == 1 else 'No'
        print(f"\n--- Segmen: Musim {season}, Diskon: {discount_status_str} ---")
        print(f"  Tidak ada aturan asosiasi dengan lift > 1.0 ditemukan untuk segmen ini.")

Menampilkan frequent itemsets multi-item (tanpa 'Toothpaste') untuk setiap segmen dengan aturan lift > 1.0:

--- Segmen: Musim Spring, Diskon: Yes ---
Frequent Itemsets dengan 2 atau Lebih Produk (Tanpa 'Toothpaste', Top 10):
+-------------------------------------------+----+
|items                                      |freq|
+-------------------------------------------+----+
|[Sponges, Plant Fertilizer]                |185 |
|[Carrots, Baby Wipes]                      |184 |
|[Butter, Baby Wipes]                       |183 |
|[Extension Cords, Onions]                  |181 |
|[Deodorant, Soap]                          |179 |
|[Apple, Rice]                              |179 |
|[Vacuum Cleaner, Feminine Hygiene Products]|179 |
|[Chips, Cheese]                            |178 |
|[Yogurt, BBQ Sauce]                        |178 |
|[Water, Butter]                            |177 |
+-------------------------------------------+----+
only showing top 10 rows


--- Segmen: Musim Spring, Diskon:

### Menunjukkan FP-Growth Menghasilkan Aturan dengan `Lift > 1.0` untuk Setiap Segmen


In [34]:
from pyspark.sql.functions import col

print("\n--- Menampilkan Aturan Asosiasi dengan Lift Tertinggi (> 1.0) untuk Setiap Segmen ---")

# Iterate through all segments where high lift rules were found
for (season, discount_applied), rules_df in high_lift_rules_by_segment.items():
    if rules_df is not None and rules_df.count() > 0:
        discount_status_str = 'Ya' if discount_applied == 1 else 'Tidak'
        print(f"\nSegmen: Musim {season}, Diskon: {discount_status_str}")
        print(f"Total aturan dengan lift > 1.0 untuk segmen ini: {rules_df.count()}")
        print("Aturan Asosiasi (Top 20, diurutkan berdasarkan lift menurun):")
        # Sort and show top 20 rules for better readability
        rules_df.orderBy(col("lift").desc()).show(20, truncate=False)
    else:
        discount_status_str = 'Ya' if discount_applied == 1 else 'Tidak'
        print(f"\nTidak ada aturan asosiasi dengan nilai lift lebih besar dari 1.0 untuk segmen Musim {season}, Diskon: {discount_status_str}.")


--- Menampilkan Aturan Asosiasi dengan Lift Tertinggi (> 1.0) untuk Setiap Segmen ---

Segmen: Musim Spring, Diskon: Ya
Total aturan dengan lift > 1.0 untuk segmen ini: 16
Aturan Asosiasi (Top 20, diurutkan berdasarkan lift menurun):
+------------------+------------------+--------------------+------------------+---------------------+
|antecedent        |consequent        |confidence          |lift              |support              |
+------------------+------------------+--------------------+------------------+---------------------+
|[Baby Wipes]      |[Carrots]         |0.043632914394119045|1.0478782275692806|0.0018311737425608567|
|[Carrots]         |[Baby Wipes]      |0.04397705544933078 |1.0478782275692804|0.0018311737425608567|
|[Baby Wipes]      |[Butter]          |0.04339577898980318 |1.0426816509931618|0.0018212217113512867|
|[Butter]          |[Baby Wipes]      |0.04375896700143472 |1.0426816509931618|0.0018212217113512867|
|[Butter]          |[Water]           |0.0423242467